In [ ]:
import pandas as pd
import numpy as np
import os
from glob import glob

parquet_files = glob('/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/**/*.parquet', recursive=True)

# funtion to reduce memory usage
def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

def drop_cols(df):
    """Clean the DataFrame by dropping specified columns and handling missing values."""
    # List of columns to drop
    columns_to_drop = ['feature_27', 'feature_00', 'feature_01', 'feature_02', 
                      'feature_03', 'feature_04', 'feature_21', 'feature_26', 
                      'feature_31', 'responder_0', 'responder_1', 'responder_2', 
                      'responder_3', 'responder_4', 'responder_5', 'responder_7', 
                      'responder_8']
    
    # Drop only existing columns
    existing_columns = [col for col in columns_to_drop if col in df.columns]
    if existing_columns:
        df = df.drop(columns=existing_columns)
    
    return df

def fill_nans(df):
    """Memory efficient version of NaN filling"""
    # Sort inplace
    df.sort_values(['date_id', 'time_id'], inplace=True)
    
    # Get numeric and non-numeric columns once
    numeric_cols = df.select_dtypes(include=['int', 'float']).columns
    other_cols = df.select_dtypes(exclude=['int', 'float']).columns
    
    # Process each symbol_id separately without groupby
    for symbol in df['symbol_id'].unique():
        mask = df['symbol_id'] == symbol
        
        # Handle numeric columns
        if len(numeric_cols) > 0:
            df.loc[mask, numeric_cols] = df.loc[mask, numeric_cols].interpolate(
                method='linear', 
                limit_direction='both',
                inplace=False
            )
        
        # Handle non-numeric columns
        for col in other_cols:
            mode_val = df.loc[mask, col].mode()
            if not mode_val.empty:
                df.loc[mask, col] = df.loc[mask, col].fillna(mode_val.iloc[0])
    
    return df

for i, file in enumerate(sorted(parquet_files)):
    df = pd.read_parquet(file)
    df = reduce_mem_usage(df)
    df = drop_cols(df)
    df = fill_nans(df)
    df.to_parquet(f'/kaggle/working/part{i}.parquet')

files = sorted(glob('/kaggle/working/*.parquet'))
df = pd.concat([pd.read_parquet(file) for file in files])
df.to_parquet('/kaggle/working/train.parquet', index=False)

Memory usage of dataframe is 654.51 MB
Memory usage after optimization is: 435.72 MB
Decreased by 33.4%
Memory usage of dataframe is 944.04 MB
Memory usage after optimization is: 548.24 MB
Decreased by 41.9%
Memory usage of dataframe is 1022.35 MB
Memory usage after optimization is: 593.72 MB
Decreased by 41.9%
Memory usage of dataframe is 1352.24 MB
Memory usage after optimization is: 693.36 MB
Decreased by 48.7%
Memory usage of dataframe is 1690.96 MB
Memory usage after optimization is: 867.04 MB
Decreased by 48.7%
Memory usage of dataframe is 1800.46 MB
Memory usage after optimization is: 923.18 MB
Decreased by 48.7%
Memory usage of dataframe is 2088.53 MB
Memory usage after optimization is: 1070.89 MB
Decreased by 48.7%
Memory usage of dataframe is 2132.85 MB
Memory usage after optimization is: 1093.61 MB
Decreased by 48.7%
Memory usage of dataframe is 2067.02 MB
Memory usage after optimization is: 1059.86 MB
Decreased by 48.7%
Memory usage of dataframe is 2112.32 MB
Memory usage a

In [3]:
df = pd.read_parquet('./kaggle/working/train.parquet')
df.head()

/Users/johnny/Library/CloudStorage/OneDrive-Personal/py/JaneStreet2024/venv/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/Users/johnny/Library/CloudStorage/OneDrive-Personal/py/JaneStreet2024/venv/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,date_id,time_id,symbol_id,weight,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,...,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_6
0,0,0,1,3.888672,0.851074,0.242920,0.263428,-0.891602,11,7,...,-1.022461,0.152222,-0.659668,-0.355713,-0.356689,-0.261475,-0.211426,-0.335449,-0.281494,0.775879
1,0,0,7,1.371094,0.676758,0.151978,0.192505,-0.521973,11,7,...,-1.052734,-0.393799,-0.741699,-0.339600,-0.331543,-0.281250,-0.182861,-0.245605,-0.302490,0.703613
2,0,0,9,2.285156,1.056641,0.187256,0.249878,-0.772949,11,7,...,-0.863281,-0.241943,-0.709961,-0.251953,-0.266357,0.377197,0.300781,-0.106812,-0.096802,2.109375
3,0,0,10,0.690430,1.139648,0.273438,0.306641,-1.262695,42,5,...,-0.530762,4.765625,0.571777,-0.287598,-0.315186,-0.226929,-0.251465,-0.215576,-0.296143,1.114258
4,0,0,14,0.440674,0.955078,0.262451,0.344482,-0.613770,44,3,...,-1.141602,0.099609,-0.662109,-0.501953,1.057617,3.677734,2.792969,2.619141,3.417969,-3.572266
